In [1]:
import pandas as pd
import geopandas as gpd



In [2]:
arrets_gdf = gpd.read_file("../raw/arrets-lignes.geojson")
quartiers_gdf = gpd.read_file("../raw/quartier_paris.geojson")

In [3]:
arrets_gdf.head()

,id,route_long_name,stop_id,stop_name,stop_lon,stop_lat,operatorname,shortname,bookingrules,mode,nom_commune,code_insee,geometry
0,IDFM:C02636,Tzen4,IDFM:426760,Charles Robin,2.466885535426266,48.61321828071897,TISSE,Tzen4,NaN,Bus,Corbeil-Essonnes,91174,POINT (2.46689 48.61322)
1,IDFM:C02636,Tzen4,IDFM:3859,Lisière des Deux Parcs,2.437547444124736,48.6241385272201,TISSE,Tzen4,NaN,Bus,Évry-Courcouronnes,91228,POINT (2.43755 48.62414)
2,IDFM:C02636,Tzen4,IDFM:487114,Les Miroirs,2.428818670322126,48.63197680175785,TISSE,Tzen4,NaN,Bus,Évry-Courcouronnes,91228,POINT (2.42882 48.63198)
3,IDFM:C02636,Tzen4,IDFM:422528,Marchais Guesdon,2.4123899036159604,48.63270646476739,TISSE,Tzen4,NaN,Bus,Évry-Courcouronnes,91228,POINT (2.41239 48.63271)
4,IDFM:C02636,Tzen4,IDFM:487111,La Mare à Pilatre,2.403606236307223,48.64252152194464,TISSE,Tzen4,NaN,Bus,Ris-Orangis,91521,POINT (2.40361 48.64252)


In [4]:
quartiers_gdf = quartiers_gdf.to_crs(arrets_gdf.crs)

In [5]:
arrets_quartiers = gpd.sjoin(
    arrets_gdf,
    quartiers_gdf,
    how="inner",
    predicate="within"
)

In [6]:
arrets_quartiers.head()

,id,route_long_name,stop_id,stop_name,stop_lon,stop_lat,operatorname,shortname,bookingrules,mode,...,c_qu,c_quinsee,l_qu,c_ar,n_sq_ar,perimetre,surface,geom_x_y,st_area_shape,st_perimeter_shape
50,IDFM:C02659,N123,IDFM:427539,Denfert-Rochereau - Arago,2.3330963199118546,48.83491703318431,Transdev Cœur Essonne,N123,NaN,Bus,...,53,7511401,Montparnasse,14,750000014,4565.136189,1.126205e+06,"{'lon': 2.331783822887114, 'lat': 48.837623095...",1.126205e+06,4564.840667
2526,IDFM:C01224,210,IDFM:37144,Tremblay - Pépinière,2.448816911248994,48.84247628550012,RATP Cap Boucles de Marne,210,NaN,Bus,...,45,7511201,Bel-Air,12,750000012,18427.822238,5.970921e+06,"{'lon': 2.4331784419569793, 'lat': 48.83799564...",5.970921e+06,18426.599903
2529,IDFM:C01224,210,IDFM:26011,Porte Jaune,2.460859284026608,48.83877186114159,RATP Cap Boucles de Marne,210,NaN,Bus,...,45,7511201,Bel-Air,12,750000012,18427.822238,5.970921e+06,"{'lon': 2.4331784419569793, 'lat': 48.83799564...",5.970921e+06,18426.599903
2582,IDFM:C01229,215,IDFM:25723,Gare de Bercy,2.38153575374016,48.84003387061312,RATP,215,NaN,Bus,...,47,7511203,Bercy,12,750000012,6155.005036,1.902932e+06,"{'lon': 2.3862100842120184, 'lat': 48.83520904...",1.902932e+06,6154.591387
2583,IDFM:C01229,215,IDFM:463522,Mairie du 12ème,2.38751831562363,48.84108992312784,RATP,215,NaN,Bus,...,46,7511202,Picpus,12,750000012,18261.910318,7.205014e+06,"{'lon': 2.428826815083849, 'lat': 48.830359242...",7.205014e+06,18260.602283


In [7]:
arrets_quartiers.columns

Index(['id', 'route_long_name', 'stop_id', 'stop_name', 'stop_lon', 'stop_lat',
       'operatorname', 'shortname', 'bookingrules', 'mode', 'nom_commune',
       'code_insee', 'geometry', 'index_right', 'n_sq_qu', 'c_qu', 'c_quinsee',
       'l_qu', 'c_ar', 'n_sq_ar', 'perimetre', 'surface', 'geom_x_y',
       'st_area_shape', 'st_perimeter_shape'],
      dtype='str')

In [10]:
arrets_quartiers = arrets_quartiers[["id","route_long_name","stop_id","stop_name","mode","c_qu","l_qu"]]

In [14]:
arrets_quartiers.to_parquet("../silver/silver_arrets_quartiers.parquet")

In [15]:
df_silver = pd.read_parquet("../silver/silver_arrets_quartiers.parquet")

In [16]:
df_aggregated = (df_silver.groupby(["l_qu","c_qu","mode"]).agg(nb_arrets=("id","count"),nb_lignes=("id","nunique")).reset_index())

In [17]:
df_aggregated.head()

,l_qu,c_qu,mode,nb_arrets,nb_lignes
0,Amérique,75,Bus,81,14
1,Amérique,75,Metro,16,3
2,Amérique,75,Tramway,6,1
3,Archives,11,Bus,12,5
4,Archives,11,Metro,2,1


In [18]:
df_aggregated.to_parquet("../gold/nb_arrets_lignes_quartiers.parquet")